# pedido-extractor-ptbr: QLoRA no Colab/Kaggle
Runtime: GPU T4. Ordem: instalar → exportar dados → avaliar base → treinar → avaliar LoRA → relatório → publicar no Hugging Face.

Resultados e adapter são salvos no Google Drive (`MyDrive/pedido-extractor`), então sobrevivem a quedas do runtime. Se reconectar, rode a célula 1 de novo e depois só as etapas que faltam: cada uma pula se o resultado já existe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/pedido-extractor'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d pedido-extractor-ptbr || git clone https://github.com/pedrohbenetton/pedido-extractor-ptbr.git
%cd /content/pedido-extractor-ptbr
!pip install -q -e '.[train]'

In [ ]:
!python -m pedido_extractor.export --out data/synthetic --n 6000 --seed 0

In [ ]:
!test -f {DRIVE}/results/base.json || python -m pedido_extractor.eval_hf --name base --out {DRIVE}/results

In [ ]:
!test -f {DRIVE}/lora/adapter_config.json || python -m pedido_extractor.train --data-dir data/synthetic --out-dir {DRIVE}/lora --epochs 2

In [ ]:
!test -f {DRIVE}/results/lora.json || python -m pedido_extractor.eval_hf --name lora --adapter {DRIVE}/lora --out {DRIVE}/results

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('src'))  # o kernel não vê pacotes instalados depois que iniciou
from pedido_extractor.evaluation import render_report

res = {n: json.load(open(f'{DRIVE}/results/{n}.json')) for n in ['base', 'lora']}
print(render_report(res))

## Publicar o adapter
Faça login com um token de escrita do Hugging Face (`notebook_login()`), troque `SEU_USUARIO` e rode.

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()
HfApi().create_repo('SEU_USUARIO/pedido-extractor-ptbr-lora', exist_ok=True)
HfApi().upload_folder(folder_path=f'{DRIVE}/lora', repo_id='SEU_USUARIO/pedido-extractor-ptbr-lora')